In [1]:
# Build a system which can able to convert one programming language code to another programming language which can be executed

# ============================================================
# Code Converter Agent (Language → Language) with Validation
# ============================================================

from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langchain.chat_models import init_chat_model


# ============================================================
# 1. Define State
# ============================================================

class CodeState(TypedDict):
    source_code: str
    source_lang: str
    target_lang: str
    converted_code: str
    validation_result: str
    retry: int


# ============================================================
# 2. Initialize Model
# ============================================================

llm = init_chat_model(
    model="llama-3.1-8b-instant",
    model_provider="groq",
    temperature=0
)


# ============================================================
# 3. Code Generator Node
# ============================================================

def generate_code(state: CodeState):

    prompt = f"""
    Convert the following {state['source_lang']} code into {state['target_lang']} code.

    Requirements:
    - Must be executable
    - Correct syntax
    - Include imports if needed
    - Only output code, no explanation

    Source Code:
    {state['source_code']}
    """

    response = llm.invoke(prompt)

    state["converted_code"] = response.content

    return state


# ============================================================
# 4. Code Validator Node
# ============================================================

def validate_code(state: CodeState):

    prompt = f"""
    Check if this {state['target_lang']} code is correct and executable.

    Code:
    {state['converted_code']}

    Reply ONLY:
    VALID
    or
    INVALID: <reason>
    """

    response = llm.invoke(prompt)

    state["validation_result"] = response.content

    return state


# ============================================================
# 5. Self Correction Node
# ============================================================

def correct_code(state: CodeState):

    prompt = f"""
    Fix this {state['target_lang']} code.

    Error:
    {state['validation_result']}

    Code:
    {state['converted_code']}

    Return corrected executable code only.
    """

    response = llm.invoke(prompt)

    state["converted_code"] = response.content

    state["retry"] += 1

    return state


# ============================================================
# 6. Router Logic
# ============================================================

def router(state: CodeState):

    if "VALID" in state["validation_result"]:
        return "end"

    elif state["retry"] >= 3:
        return "end"

    else:
        return "correct"


# ============================================================
# 7. Build Graph
# ============================================================

graph = StateGraph(CodeState)

graph.add_node("generate", generate_code)
graph.add_node("validate", validate_code)
graph.add_node("correct", correct_code)

graph.add_edge(START, "generate")

graph.add_edge("generate", "validate")

graph.add_conditional_edges(
    "validate",
    router,
    {
        "correct": "correct",
        "end": END
    }
)

graph.add_edge("correct", "validate")


app = graph.compile()


# ============================================================
# 8. Example Run
# ============================================================

input_state = {
    "source_code": """
def add(a, b):
    return a + b

print(add(2,3))
""",
    "source_lang": "Python",
    "target_lang": "Java",
    "converted_code": "",
    "validation_result": "",
    "retry": 0
}


result = app.invoke(input_state)


# ============================================================
# 9. Output
# ============================================================

print("\nConverted Code:\n")
print(result["converted_code"])

print("\nValidation Result:\n")
print(result["validation_result"])


E:\EDU_CARE\arg_venv\Lib\site-packages\langchain_core\_api\deprecation.py:26: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1
E:\EDU_CARE\arg_venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



Converted Code:

```java
public class Main {
    public static void main(String[] args) {
        System.out.println(add(2, 3));
    }

    public static int add(int a, int b) {
        return a + b;
    }
}
```

Validation Result:

VALID
